# Import libraries

In [1]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load full ground truth

In [2]:
def load_gt(gt_fname):
    with open(gt_fname) as f:
        gt = json.load(f)

    rows_gt = []
    for video_id, info in gt["database"].items():
        duration = info["duration"]
        subset = info["subset"]
        for s in info["annotations"]:
            rows_gt.append(
                {
                    "video_id": video_id,
                    "duration": duration,
                    "subset": subset,
                    "start": s["segment"][0],
                    "end": s["segment"][1],
                    "label": s["label"],
                }
            )

    return pd.DataFrame(rows_gt)

In [3]:
gt_df = load_gt("../data/charades/annotations/charades.json")

# Load EXO_EGO annotations

In [4]:
training_ego_df = pd.read_csv("../data/charades/annotations/CharadesEgo_v1_train_only1st.csv")
testing_ego_df  = pd.read_csv("../data/charades/annotations/CharadesEgo_v1_test_only1st.csv")
# prepare to stack training and testing splits
training_ego_df = training_ego_df[['id', 'charades_video']]
testing_ego_df = testing_ego_df[['id', 'charades_video']]
ego_df = pd.concat([training_ego_df, testing_ego_df], axis=0)
# clean ego_df: remove rows with na value
ego_df = ego_df.dropna()
ego_df.head()

,id,charades_video
0,D3TR8EGO,1K0SU
3,6U5TEEGO,YFZRG
4,KCSBQEGO,SKUOZ
5,F6XROEGO,V7T91
7,28DG6EGO,52MV9


# Sample the 10 most frequent classes (~ 200 segments for each class)

In [5]:
# Select the 10 most common classes
most_common_classes = sorted(
    gt_df["label"].value_counts()[:10].index.tolist()
)  

# Filter the original dataset to include only rows with classes in "most common classes"
most_common_gt = gt_df[gt_df["label"].isin(most_common_classes)].copy()

"""Select training dataset"""
# Sample ~200 rows per label from the filtered dataset
most_common_rows_training = most_common_gt.groupby('label', group_keys=False).sample(n=200, random_state=42).copy()
# Get all unique video_ids from those sampled rows
video_ids_to_include_training = most_common_rows_training["video_id"].unique()
# Filter the original df to include all rows with those video_ids
most_common_gt_training = most_common_gt[most_common_gt["video_id"].isin(video_ids_to_include_training)].copy()

"""Select testing dataset"""
# Get the rest of most_common_gt
rest_most_common_gt = most_common_gt[~most_common_gt["video_id"].isin(video_ids_to_include_training)].copy()
# Sample ~200 rows per label from the filtered dataset
most_common_rows_testing = rest_most_common_gt.groupby('label', group_keys=False).sample(n=200, random_state=42).copy()
# Get all unique video_ids from those sampled rows
video_ids_to_include_testing = most_common_rows_testing["video_id"].unique()
# Filter the original df to include all rows with those video_ids
most_common_gt_testing = rest_most_common_gt[rest_most_common_gt["video_id"].isin(video_ids_to_include_testing)].copy()


In [6]:
most_common_gt_training.shape

(4613, 6)

In [7]:
most_common_gt_testing.shape

(4067, 6)

In [8]:
most_common_gt_training.merge(most_common_gt_testing, on='video_id', how='inner')

,video_id,duration_x,subset_x,start_x,end_x,label_x,duration_y,subset_y,start_y,end_y,label_y


# Enrich information of most_sample_gt_df to get relevant ego_id

In [9]:
df_exo_ego_training = most_common_gt_training.merge(ego_df, left_on='video_id', right_on='charades_video', how='inner').rename(columns={'id': 'ego_id'})
df_exo_ego_testing = most_common_gt_testing.merge(ego_df, left_on='video_id', right_on='charades_video', how='inner').rename(columns={'id': 'ego_id'})

In [10]:
df_exo_ego_training.shape

(1851, 8)

In [11]:
df_exo_ego_training.sample(5)

,video_id,duration,subset,start,end,label,ego_id,charades_video
1366,RZIAJ,30.21,training,19.0,25.0,Someone is standing up from somewhere,UVYKREGO,RZIAJ
1390,NALYZ,33.58,training,18.0,26.2,Drinking from a cup/glass/bottle,N8N2AEGO,NALYZ
842,ORAT0,30.62,training,5.8,11.8,Someone is going from standing to sitting,09O6SEGO,ORAT0
1258,SG2HN,30.58,training,0.0,32.0,Holding a phone/camera,GWXIBEGO,SG2HN
845,OLEWM,30.04,training,0.0,9.7,Someone is eating something,LO6XQEGO,OLEWM


In [12]:
df_exo_ego_testing.shape

(1513, 8)

In [13]:
df_exo_ego_testing.sample(5)

,video_id,duration,subset,start,end,label,ego_id,charades_video
1300,HD38O,34.42,training,0.0,16.7,Holding a phone/camera,EUDKHEGO,HD38O
1484,ISTQI,30.58,training,11.3,19.1,Someone is going from standing to sitting,DI2PBEGO,ISTQI
947,LW1W2,30.71,training,8.1,16.0,Holding a cup/glass/bottle of something,IYPROEGO,LW1W2
698,KK8N9,12.17,training,8.9,13.0,Someone is smiling,GSD40EGO,KK8N9
1068,XIUQJ,35.08,training,0.0,11.0,Drinking from a cup/glass/bottle,6BSIJEGO,XIUQJ


# Start building dataset

In [14]:
def build_exo_ego_dataset(
    sample_gt_df,
    category_idx_fname="../data/charades/annotations/sample_category_idx.txt",
    gt_fname="../data/charades/annotations/sample_charades.json",
    haveExo=True,
    haveEgo=True,
    subset="training"
):
    # Save category index to a text file
    actions = sorted(sample_gt_df["label"].unique().tolist())
    with open(category_idx_fname, "w") as f:
        for action in actions:
            f.write(action + "\n")

    # Save sample annotations to JSON file
    output = {"version": "Most_Exo_Ego", "database": {}}

    for video_id, group in sample_gt_df.groupby("video_id"):
        # keep duration/subset from the original gt
        duration = sample_gt_df[sample_gt_df["video_id"] == video_id]["duration"].iloc[0]
        subset = subset
        ego_id = sample_gt_df[sample_gt_df["video_id"] == video_id]["ego_id"].iloc[0]
        exo_id = ego_id[:-3]

        annotations = []
        for _, row in group.iterrows():
            annotations.append(
                {"segment": [row["start"], row["end"]], "label": row["label"]}
            )

        if haveEgo:
            # for ego view extraction
            output["database"][ego_id] = {
                "video_id": video_id,  # not sure if necessary
                "duration": duration,
                "subset": subset,
                "annotations": annotations,
            }

        if haveExo:
            # for exo view extraction
            output["database"][exo_id] = {
                "video_id": video_id,  # not sure if necessary
                "duration": duration,
                "subset": subset,
                "annotations": annotations,
            }

    with open(gt_fname, "w") as f:
        json.dump(output, f, indent=2)

In [15]:
# build_exo_ego_dataset(
#     df_exo_ego_training,
#     category_idx_fname="../data/charades/annotations/exo_only_training_category_idx.txt",
#     gt_fname="../data/charades/annotations/exo_only_training_charades.json",
#     haveEgo=False,
#     haveExo=True,
#     subset="training"
# )

In [16]:
build_exo_ego_dataset(
    df_exo_ego_testing,
    category_idx_fname="../data/charades/annotations/ego_only_testing_category_idx.txt",
    gt_fname="../data/charades/annotations/ego_only_testing_charades.json",
    haveEgo=True,
    haveExo=False,
    subset="testing"
)